# VitroVision SAM3 Cascade API
Pipeline: tap → detect bottle → crop ROI → segment plant → parameters

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
!pip install -q transformers torch torchvision opencv-python pillow numpy fastapi uvicorn nest-asyncio pyngrok

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageDraw
import cv2
from transformers import Sam3Processor, Sam3Model
from io import BytesIO
import base64

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("SAM3 ready")

In [ ]:
PIXEL_TO_CM = 0.1  # placeholder, calibrate later

def detect_bottles(image_pil, device):
    inputs = processor(images=image_pil, text="glass jar bottle", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    h, w = image_pil.height, image_pil.width
    results = processor.post_process_instance_segmentation(
        outputs, threshold=0.3, mask_threshold=0.3, target_sizes=[[h,w]]
    )[0]
    return results["masks"], results.get("scores", [])

def match_tap_to_bottle(masks, scores, tap_x, tap_y):
    best_idx, best_dist = -1, float("inf")
    for i in range(len(masks)):
        m = masks[i].cpu().numpy().astype(bool)
        if m[tap_y, tap_x]:  # tap inside this mask
            return i
        # else find closest mask pixel
        ys, xs = np.where(m)
        if len(xs) > 0:
            dist = np.min(np.sqrt((xs - tap_x)**2 + (ys - tap_y)**2))
            if dist < best_dist:
                best_dist = dist
                best_idx = i
    return best_idx if best_dist < 200 else -1

def segment_plant_in_roi(image_pil, roi_box, prompts, device):
    x1, y1, x2, y2 = roi_box
    crop = image_pil.crop(roi_box)
    ch, cw = crop.height, crop.width

    all_results = []
    for prompt in prompts:
        inputs = processor(images=crop, text=prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        res = processor.post_process_instance_segmentation(
            outputs, threshold=0.5, mask_threshold=0.5,
            target_sizes=[[ch, cw]]
        )[0]
        all_results.append({"prompt": prompt, "masks": res["masks"], "scores": res.get("scores", [])})

    # merge all plant masks
    merged = np.zeros((ch, cw), dtype=np.uint8)
    leaf_count = 0
    for r in all_results:
        for i in range(len(r["masks"])):
            m = r["masks"][i].cpu().numpy().astype(np.uint8)
            merged = cv2.bitwise_or(merged, m)
            if "leaf" in r["prompt"] or "plant" in r["prompt"]:
                leaf_count += 1

    crop_np = np.array(crop.convert("RGB"))
    crop_hsv = cv2.cvtColor(crop_np, cv2.COLOR_RGB2HSV)

    # parameters
    plant_pixels = int(np.sum(merged > 0))
    total_pixels = cw * ch
    coverage_ratio = plant_pixels / total_pixels if total_pixels > 0 else 0

    rows = np.any(merged, axis=1)
    cols = np.any(merged, axis=0)
    if rows.any():
        plant_top = int(np.argmax(rows))
        plant_bot = int(ch - np.argmax(rows[::-1]))
        height_px = plant_bot - plant_top
    else:
        height_px = 0
    if cols.any():
        plant_left = int(np.argmax(cols))
        plant_right = int(cw - np.argmax(cols[::-1]))
        width_px = plant_right - plant_left
    else:
        width_px = 0

    plant_region = crop_hsv[merged > 0]
    mean_green = float(np.mean(crop_np[merged > 0, 1])) if len(plant_region) > 0 else 0
    mean_hue = float(np.mean(plant_region[:, 0])) if len(plant_region) > 0 else 0

    # encode mask (on full image coords)
    full_mask = np.zeros((image_pil.height, image_pil.width), dtype=np.uint8)
    full_mask[y1:y2, x1:x2] = merged
    _, mask_bytes = cv2.imencode(".png", full_mask * 255)
    mask_b64 = base64.b64encode(mask_bytes).decode("utf-8")

    # draw overlay preview
    vis = np.array(image_pil.convert("RGB")).copy()
    overlay = np.zeros_like(vis)
    overlay[full_mask > 0] = [0, 200, 0]
    vis = cv2.addWeighted(vis, 0.7, overlay, 0.3, 0)
    # draw roi box
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
    _, preview_bytes = cv2.imencode(".png", cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    preview_b64 = base64.b64encode(preview_bytes).decode("utf-8")

    return {
        "leaf_count_approx": leaf_count,
        "coverage_ratio": round(coverage_ratio, 4),
        "height_cm": round(height_px * PIXEL_TO_CM, 2),
        "width_cm": round(width_px * PIXEL_TO_CM, 2),
        "height_px": height_px,
        "width_px": width_px,
        "mean_greenness": round(mean_green, 2),
        "mean_hue": round(mean_hue, 2),
        "total_plant_area_px": plant_pixels,
        "roi_box": [x1, y1, x2, y2],
        "mask_png_b64": mask_b64,
        "preview_png_b64": preview_b64,
    }

def cascade_process(image_pil, tap_x, tap_y, prompts=None, device=device):
    if prompts is None:
        prompts = ["leaf", "plant", "stem"]

    # step 1: detect all bottles
    bottle_masks, bottle_scores = detect_bottles(image_pil, device)
    if len(bottle_masks) == 0:
        return {"error": "no bottle detected", "results": None}

    # step 2: match tap to nearest bottle
    idx = match_tap_to_bottle(bottle_masks, bottle_scores, tap_x, tap_y)
    if idx == -1:
        return {"error": "tap outside any bottle", "results": None}

    selected_mask = bottle_masks[idx].cpu().numpy().astype(bool)

    # step 3: get ROI from bottle mask
    ys, xs = np.where(selected_mask)
    roi_box = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
    # add padding
    pad = 20
    h, w = image_pil.height, image_pil.width
    roi_box = (max(0, roi_box[0]-pad), max(0, roi_box[1]-pad),
               min(w, roi_box[2]+pad), min(h, roi_box[3]+pad))

    # step 4: segment plant inside ROI + parameters
    results = segment_plant_in_roi(image_pil, roi_box, prompts, device)
    results["bottle_index"] = int(idx)
    results["bottles_detected"] = len(bottle_masks)
    results["tap_point"] = [tap_x, tap_y]

    return {"error": None, "results": results}

print("cascade pipeline ready")

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
import nest_asyncio

app = FastAPI(title="VitroVision Cascade API")

class CascadeRequest(BaseModel):
    image: str
    tap_x: int
    tap_y: int
    prompts: Optional[List[str]] = None
    image_width: Optional[int] = None
    image_height: Optional[int] = None

@app.post("/cascade")
async def cascade(req: CascadeRequest):
    try:
        img_bytes = base64.b64decode(req.image)
        img = Image.open(BytesIO(img_bytes)).convert("RGB")
        # scale tap coordinates if resized
        tx, ty = req.tap_x, req.tap_y
        if req.image_width and req.image_height:
            sx = img.width / req.image_width
            sy = img.height / req.image_height
            tx, ty = int(tx * sx), int(ty * sy)
        result = cascade_process(img, tx, ty, req.prompts)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {"status": "ok"}

nest_asyncio.apply()
print("API ready")

In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8000).public_url
print(f"Cascade API: {public_url}/cascade")
print(f"Health:      {public_url}/health")

In [ ]:
uvicorn.run(app, host="0.0.0.0", port=8000)